# GPU checkpoint and restore

This notebook checks that pausing a Workspace **checkpoints live GPU state** and that resuming it
restores that exact state — rather than just restarting the pod and replaying the notebook.

The trick is to build state that *cannot be recreated by re-running the cells*: a random value
generated in device memory, held only by this Python process. If it is still there, byte for byte,
after a pause and resume, then the CUDA context and its device memory really were checkpointed.

**Requirements:** a Workspace using the **GPU** image and the **GPU T4 Spot** pod config, on a
snapshot-enabled `WorkspaceKind`.

**How to run:**

1. Run step 1 and step 2 below.
2. Pause the Workspace and wait for it to finish, then resume it.
3. Reopen this notebook and run **only** step 3. Do not re-run steps 1–2 — re-running them would
   rebuild the state and defeat the test.

In [ ]:
# Step 1 — confirm we actually have a GPU.
import os
import socket
import time

import torch

print(f"host       : {socket.gethostname()}")
print(f"pid        : {os.getpid()}")
print(f"torch      : {torch.__version__}")
print(f"cuda build : {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device is visible, so there is nothing to checkpoint. Recreate this "
        "Workspace with the GPU image and the 'GPU T4 Spot' pod config."
    )

DEVICE = torch.device("cuda:0")
print(f"device     : {torch.cuda.get_device_name(0)}")
print(f"capability : {torch.cuda.get_device_capability(0)}")

In [ ]:
# Step 2 — build state that only exists in this process and in GPU memory.
import hashlib
import secrets
import threading

# Seeded from the OS entropy pool, so re-running this cell can never reproduce it.
MARKER = secrets.token_hex(8)
PID = os.getpid()
CREATED_AT = time.time()

# 64 MiB of random floats generated on the device. Nothing on the home PVC can be
# used to reconstruct these bytes -- they live only in GPU memory.
torch.manual_seed(secrets.randbits(63))
GPU_STATE = torch.rand(4096, 4096, device=DEVICE)


def fingerprint(tensor):
    """A short, stable hash of a tensor's contents."""
    return hashlib.sha256(tensor.detach().cpu().numpy().tobytes()).hexdigest()[:16]


FINGERPRINT = fingerprint(GPU_STATE)

# A thread that keeps counting while the process runs. Comparing its count against the
# wall clock after the resume tells us how long the process was actually frozen.
TICKS = 0


def _tick():
    global TICKS
    while True:
        time.sleep(1)
        TICKS += 1


threading.Thread(target=_tick, daemon=True).start()

print(f"marker      : {MARKER}")
print(f"pid         : {PID}")
print(f"gpu tensor  : {tuple(GPU_STATE.shape)} {GPU_STATE.dtype} on {GPU_STATE.device}")
print(f"size        : {GPU_STATE.element_size() * GPU_STATE.nelement() / 2**20:.0f} MiB")
print(f"fingerprint : {FINGERPRINT}")
print(f"allocated   : {torch.cuda.memory_allocated() / 2**20:.0f} MiB on device")

## Now pause the Workspace

Note the `fingerprint` printed above, then pause — either with the **Pause** button in the Workspaces
UI, or from a terminal with access to the cluster:

```bash
kubectl -n <namespace> patch workspace <name> --type=merge -p '{"spec":{"paused":true}}'
```

The pause is not instant. The addon holds `spec.paused=false` until GKE has taken the snapshot, so
watch for it to settle:

```bash
kubectl -n <namespace> get workspace <name> \
  -o jsonpath='{.spec.paused}{" "}{.metadata.annotations.podsnapshot\.gke\.kubeflow\.org/checkpoint-state}{"\n"}'
# -> "true Ready" once the checkpoint is safely in GCS
```

Then resume it (**Resume** in the UI, or set `spec.paused` back to `false`), wait for the pod to be
`1/1`, reopen this notebook, and run step 3 below **without re-running steps 1 and 2**.

In [ ]:
# Step 3 — run this AFTER the resume. Do not re-run the cells above.
try:
    MARKER, PID, CREATED_AT, GPU_STATE, FINGERPRINT
except NameError:
    raise RuntimeError(
        "The state from step 2 is gone, so this is a fresh kernel: the pod was restarted "
        "rather than restored. GPU checkpoint/restore FAILED."
    ) from None

checks = []

current_pid = os.getpid()
checks.append(("same process", current_pid == PID, f"pid {current_pid}, was {PID}"))

restored = fingerprint(GPU_STATE)
checks.append(
    ("gpu memory intact", restored == FINGERPRINT, f"{restored}, was {FINGERPRINT}")
)

# Surviving bytes are not enough: the CUDA context has to be usable again too.
try:
    probe = (GPU_STATE @ GPU_STATE.T).sum().item()
    torch.cuda.synchronize()
    usable, detail = True, f"fresh matmul on {GPU_STATE.device} -> {probe:.6e}"
except Exception as exc:  # noqa: BLE001 - we want to report any failure mode
    usable, detail = False, f"{type(exc).__name__}: {exc}"
checks.append(("cuda context usable", usable, detail))

# Wall clock that elapsed without the ticker advancing is time the process spent frozen.
frozen = time.time() - CREATED_AT - TICKS
checks.append(("process was frozen", frozen > 5, f"~{frozen:.0f}s paused, {TICKS}s running"))

label_width = max(len(name) for name, _, _ in checks)
for name, ok, detail in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {name.ljust(label_width)}   {detail}")

print()
print("GPU checkpoint/restore: " + ("PASS" if all(ok for _, ok, _ in checks) else "FAIL"))

## Reading the result

| Failure | What it means |
|---|---|
| `NameError` / fresh kernel | The pod was recreated, not restored. Either the Workspace is not snapshot-enabled, or the restore did not happen. |
| `same process` fails | A new process took over. Same conclusion as above. |
| `gpu memory intact` fails | The process survived but device memory did not come back identical — a genuine GPU checkpoint bug. |
| `cuda context usable` fails | The bytes survived but the CUDA context did not rebind to the device after restore. |
| `process was frozen` fails | Nothing was actually paused; check that the pause completed before you resumed. |

To watch the machinery from a terminal with cluster access:

```bash
# The snapshot taken for this Workspace, and where it went.
kubectl -n <namespace> get podsnapshots

# What the addon decided, live.
kubectl -n kubeflow-workspaces logs -l app=gke-workspace-snapshot-addon -f --prefix
```

> **If the pod never leaves `Pending`:** snapshot-enabled pods are mutated to run under
> `runtimeClassName=gvisor`, so the node has to offer both a T4 and the gVisor sandbox. A GPU node
> pool without gVisor enabled will never satisfy that, and the pod will sit unschedulable.
> `kubectl -n <namespace> describe pod <pod>` will say so.